# Recurring Operations: Group Comparison Analysis

Methods covered:
  1. Kaplan-Meier + log-rank (time to first re-operation)
  2. Cox Proportional Hazards (time to first re-operation)
  3. Andersen-Gill model (all re-operations, independent increments)
  4. Prentice-Williams-Peterson (PWP) model (ordered recurring events)
  5. Negative Binomial regression (total count of re-operations)
  6. Mean Cumulative Function (MCF) plot

# Customization

In [ ]:
# ---- 0. Install / load packages ----
pkgs <- c(
    "survival", "survminer", "MASS", "dplyr", "ggplot2", "tibble",
    "readxl", "lubridate", "ggbeeswarm"
)
new_pkgs <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
if (length(new_pkgs)) install.packages(new_pkgs)
invisible(lapply(pkgs, library, character.only = TRUE))

set.seed(42)


In [ ]:
# ---- Plot dimensions (width x height in inches) ----
options(repr.plot.width = 12, repr.plot.height = 9)
options(warn = -1)


# LOAD DATA
Read from `lor.xlsx`, sheet **"данные"**.

Columns used:
- `группа`
- `id` — идентификатор пациента
- `возраст`
- `пол`
- `длительность заболевания`
- `удаление полипов в анамнезе` — количество предыдущих удалений полипов
- `дата операции` — дата индексной операции
- `оперированные пазухи` — количество оперированных пазух
- `операция 1` / `дата операции 1` — 1-я повторная операция (да/нет + дата)
- `операция 2` / `дата операции 2` — 2-я повторная операция (да/нет + дата)

Длительность наблюдения измеряется в **днях** с момента индексной операции.  
Пациенты без последующего события цензурированы на последнюю дату, наблюдаемую в наборе данных.  
Для удобства используется константа `six_months_days = 365.25 / 2 ≈ 182.6` для интервалов на оси.


In [ ]:
# ---- Read raw data ----
raw <- read_excel(
  "C:\\Analysis\\OTOLARING\\Nidelko\\lor.xlsx",
  sheet = "данные"
)


In [ ]:
stop_date <- as.Date("2026-02-01")


In [ ]:
# ---- Helper: days between two dates ----
days_diff <- function(d_start, d_end) {
  as.numeric(difftime(as.Date(d_end), as.Date(d_start), units = "days"))
}

# ---- Select and rename columns ----
data <- raw |>
  transmute(
    id = as.character(`id`),
    group = factor(`группа`),
    age = as.integer(`возраст`),
    sex = `пол`,
    ill_dur = as.integer(`длительность заболевания`),
    polyps = `удаление полипов в анамнезе`,
    date_operation = as.Date(`дата операции`),
    sinuses = `оперированные пазухи`,
    followup = days_diff(as.Date(`дата операции`), stop_date),
    reop1 = ifelse(is.na(`дата операции 1`), 0, 1),
    date_reoperation1 = as.Date(`дата операции 1`),
    reop2 = ifelse(is.na(`дата операции 2`), 0, 1),
    date_reoperation2 = as.Date(`дата операции 2`)
  )

# 6 months in days (used for axis breaks)
six_months_days <- 365.25 / 2 # 182.625

# ---- Study end = latest date observed across all patients ----
study_end <- max(c(data$date_reoperation2, data$date_reoperation1, data$date_operation), na.rm = TRUE)
cat(sprintf("Study end date (latest observed): %s\n\n", as.character(study_end)))

# ---- Patient-level summary (one row per patient) ----
patient_df <- data |>
  dplyr::mutate(
    n_reops  = reop1 + reop2,
    followup = days_diff(date_operation, study_end)
  )

cat(sprintf("Patients loaded: %d\n", nrow(patient_df)))
cat(sprintf("Groups: %s\n\n", paste(levels(patient_df$group), collapse = ", ")))

print(patient_df)


In [ ]:
# ---- Patient-level summary table ----
cat("=== Patient summary ===\n")
patient_df |>
  group_by(group) |>
  summarise(
    n            = n(),
    median_fup   = round(median(followup), 1),
    mean_reops   = round(mean(n_reops), 2),
    pct_any_reop = round(mean(n_reops > 0) * 100, 1)
  ) |>
  print()


In [ ]:
# ---- Derive palette from actual group levels ----
group_levels <- levels(patient_df$group)
n_groups <- length(group_levels)

# Base set of distinguishable colors; extend if more groups are added
base_colors <- c(
    "#E74C3C", "#2ECC71", "#3498DB", "#9B59B6",
    "#F39C12", "#1ABC9C", "#E67E22", "#2980B9"
)
group_palette <- setNames(base_colors[seq_len(n_groups)], group_levels)

cat(sprintf("Groups (%d): %s\n", n_groups, paste(group_levels, collapse = ", ")))
cat("Palette:\n")
print(group_palette)


In [ ]:
# ---- Combined groups: ОГ1 + ОГ2 → ОГ ----
# patient_df_og is used in all "Part 2" analyses below
patient_df_og <- patient_df |>
  dplyr::mutate(group = factor(
    ifelse(as.character(group) %in% c("ОГ1", "ОГ2"), "ОГ", as.character(group))
  ))

group_levels_og <- levels(patient_df_og$group)
n_groups_og <- length(group_levels_og)
group_palette_og <- setNames(base_colors[seq_len(n_groups_og)], group_levels_og)

cat(sprintf("Combined groups (%d): %s\n", n_groups_og, paste(group_levels_og, collapse = ", ")))
print(group_palette_og)


# DATA FORMATS

## Follow-up time description

Длительность наблюдения -- это количество дней с момента индексной операции до даты окончания исследования (административное цензурирование).

### Все группы

In [ ]:
# ---- Follow-up time: basic statistics ----
fup_stats <- function(x) {
  tibble::tibble(
    n      = length(x),
    mean   = round(mean(x), 1),
    sd     = round(sd(x), 1),
    median = round(median(x), 1),
    q1     = round(quantile(x, 0.25), 1),
    q3     = round(quantile(x, 0.75), 1),
    min    = round(min(x), 1),
    max    = round(max(x), 1)
  )
}

cat("=== Follow-up time (days) — all patients ===\n")
print(fup_stats(patient_df$followup))

cat("\n=== Follow-up time (days) — by group ===\n")
patient_df |>
  group_by(group) |>
  summarise(fup_stats(followup), .groups = "drop") |>
  print()


In [ ]:
# ---- Follow-up time: beeswarm — all patients ----

fup_all_df <- patient_df |>
  dplyr::mutate(fup_years = followup / 365.25)

y_breaks <- seq(0, ceiling(max(fup_all_df$fup_years)), by = 0.5)

fup_beeswarm_all <- ggplot(fup_all_df, aes(x = "", y = fup_years)) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 0 & reop2 == 0),
    cex = 2.5, size = 3.0, shape = 16, alpha = 0.7, colour = "grey40"
  ) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 1 | reop2 == 1),
    cex = 2.5, size = 4, shape = 17, alpha = 0.85, colour = "grey20"
  ) +
  stat_summary(
    fun.data = median_hilow, fun.args = list(conf.int = 0.5),
    geom = "crossbar", width = 0.3, linewidth = 0.6,
    colour = "black", fatten = 2
  ) +
  scale_y_continuous(breaks = y_breaks) +
  labs(
    x        = "все пациенты",
    y        = "длительность наблюдения (годы)"
  ) +
  theme_bw(base_size = 16)


print(fup_beeswarm_all)

# head(fup_all_df, n=50)


In [ ]:
# ---- Follow-up time: beeswarm — by group ----

fup_grp_df <- patient_df |>
  dplyr::mutate(fup_years = followup / 365.25)

fup_beeswarm_grp <- ggplot(fup_grp_df, aes(x = group, y = fup_years, colour = group)) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 0 & reop2 == 0),
    cex = 2.5, size = 2.0, shape = 16, alpha = 0.7
  ) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 1 | reop2 == 1),
    cex = 2.5, size = 3.8, shape = 17, alpha = 0.85
  ) +
  stat_summary(
    fun.data = median_hilow, fun.args = list(conf.int = 0.5),
    geom = "crossbar", width = 0.3, linewidth = 0.6,
    colour = "black", fatten = 2
  ) +
  scale_colour_manual(values = group_palette, guide = "none") +
  scale_y_continuous(breaks = y_breaks) +
  labs(
    x        = "группа",
    y        = "длительность наблюдения (годы)"
  ) +
  theme_bw(base_size = 16)


print(fup_beeswarm_grp)


In [ ]:
# ---- Reoperation times: beeswarm — all patients ----
# One point per reoperation event; shape distinguishes 1st vs 2nd reoperation.
# Patients who never had a reoperation are NOT plotted (nothing to show).

# ---- Follow-up time: beeswarm — all patients ----

fup_all_df <- patient_df |>
  dplyr::mutate(fup_years = ifelse(reop1 == 1, days_diff(date_operation, date_reoperation1) / 365.25, followup / 365.25))

y_breaks <- seq(0, ceiling(max(fup_all_df$fup_years)), by = 0.5)

fup_beeswarm_all <- ggplot(fup_all_df, aes(x = "", y = fup_years)) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 0 & reop2 == 0),
    cex = 2.5, size = 3.0, shape = 16, alpha = 0.7, colour = "grey40"
  ) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 1 | reop2 == 1),
    cex = 2.5, size = 4, shape = 17, alpha = 0.85, colour = "grey20"
  ) +
  stat_summary(
    fun.data = median_hilow, fun.args = list(conf.int = 0.5),
    geom = "crossbar", width = 0.3, linewidth = 0.6,
    colour = "black", fatten = 2
  ) +
  scale_y_continuous(breaks = y_breaks) +
  labs(
    x        = "все пациенты",
    y        = "длительность наблюдения (годы)"
  ) +
  theme_bw(base_size = 16)


print(fup_beeswarm_all)


In [ ]:
# ---- Reoperation times: beeswarm — by group (all groups) ----
# One point per reoperation event; shape distinguishes 1st vs 2nd reoperation.

fup_all_df <- patient_df |>
  dplyr::mutate(fup_years = ifelse(reop1 == 1, days_diff(date_operation, date_reoperation1) / 365.25, followup / 365.25))


y_breaks_reop <- seq(0, ceiling(max(fup_all_df$fup_years)), by = 0.5)

fup_beeswarm_grp <- ggplot(fup_all_df, aes(x = group, y = fup_years, colour = group)) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 0 & reop2 == 0),
    cex = 2.5, size = 2.0, shape = 16, alpha = 0.7
  ) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 1 | reop2 == 1),
    cex = 2.5, size = 3.8, shape = 17, alpha = 0.85
  ) +
  stat_summary(
    fun.data = median_hilow, fun.args = list(conf.int = 0.5),
    geom = "crossbar", width = 0.3, linewidth = 0.6,
    colour = "black", fatten = 2
  ) +
  scale_colour_manual(values = group_palette, guide = "none") +
  scale_y_continuous(breaks = y_breaks) +
  labs(
    x        = "группа",
    y        = "длительность наблюдения (годы)"
  ) +
  theme_bw(base_size = 16)


print(fup_beeswarm_grp)


### ОГ1+ОГ2 vs КГ — длительность наблюдения

In [ ]:
# ---- Follow-up time: basic statistics — ОГ combined ----
cat("=== Follow-up time (days) — ОГ combined vs КГ ===\n")
patient_df_og |>
  group_by(group) |>
  summarise(fup_stats(followup), .groups = "drop") |>
  print()


In [ ]:
# ---- Follow-up time: beeswarm — ОГ combined vs КГ ----
fup_og_df <- patient_df_og |>
  dplyr::mutate(fup_years = followup / 365.25)

y_breaks_og <- seq(0, ceiling(max(fup_og_df$fup_years)), by = 0.5)

fup_beeswarm_og <- ggplot(fup_og_df, aes(x = group, y = fup_years, colour = group)) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 0 & reop2 == 0),
    cex = 2.5, size = 2.0, shape = 16, alpha = 0.7
  ) +
  geom_beeswarm(
    data = ~ subset(., reop1 == 1 | reop2 == 1),
    cex = 2.5, size = 3.8, shape = 17, alpha = 0.85
  ) +
  stat_summary(
    fun.data = median_hilow, fun.args = list(conf.int = 0.5),
    geom = "crossbar", width = 0.3, linewidth = 0.6,
    colour = "black", fatten = 2
  ) +
  scale_colour_manual(values = group_palette_og, guide = "none") +
  scale_y_continuous(breaks = y_breaks_og) +
  labs(
    x = "группа",
    y = "длительность наблюдения (годы)"
  ) +
  theme_bw(base_size = 16)

print(fup_beeswarm_og)


In [ ]:
# ---- Reoperation times: beeswarm — ОГ combined vs КГ ----

reop_times_og_df <- bind_rows(
  patient_df_og |>
    filter(reop1 == 1) |>
    transmute(
      id, group,
      reop_year = days_diff(date_operation, date_reoperation1) / 365.25,
      reop_num = "1-я повторная операция"
    ),
  patient_df_og |>
    filter(reop2 == 1) |>
    transmute(
      id, group,
      reop_year = days_diff(date_operation, date_reoperation2) / 365.25,
      reop_num = "2-я повторная операция"
    )
) |>
  dplyr::mutate(reop_num = factor(reop_num,
    levels = c(
      "1-я повторная операция",
      "2-я повторная операция"
    )
  ))

# y axis breaks

y_breaks_reop_og <- seq(0, ceiling(max(reop_times_og_df$reop_year)), by = 0.5)


reop_beeswarm_og <- ggplot(
  reop_times_og_df,
  aes(
    x = group, y = reop_year,
    colour = group, shape = reop_num
  )
) +
  geom_beeswarm(cex = 2.5, size = 3.5, alpha = 0.85) +
  stat_summary(
    aes(group = group),
    fun.data = median_hilow, fun.args = list(conf.int = 0.5),
    geom = "crossbar", width = 0.3, linewidth = 0.6,
    colour = "black", fatten = 2
  ) +
  scale_colour_manual(values = group_palette_og, guide = "none") +
  scale_shape_manual(
    values = c(
      "1-я повторная операция" = 16,
      "2-я повторная операция" = 17
    ),
    name = NULL
  ) +
  scale_y_continuous(breaks = y_breaks_reop_og) +
  labs(
    x = "группа",
    y = "время до повторной операции (годы)"
  ) +
  theme_bw(base_size = 16) +
  theme(legend.position = "top")

print(reop_beeswarm_og)


In [ ]:
first_reop_df <- patient_df |>
  dplyr::mutate(
    time_first = ifelse(
      reop1 == 1,
      days_diff(date_operation, date_reoperation1),
      followup
    ),
    event_first = as.integer(reop1 == 1)
  )
head(first_reop_df)


In [ ]:
# ---- 2a (Part 2): Wide format — ОГ1+ОГ2 combined ----
first_reop_df_og <- patient_df_og |>
  dplyr::mutate(
    time_first = ifelse(
      reop1 == 1,
      days_diff(date_operation, date_reoperation1),
      followup
    ),
    event_first = as.integer(reop1 == 1)
  )
head(first_reop_df_og)


In [ ]:
# ---- 2b. Long (counting process) format: one row per event interval ----
# Format:  id | group | tstart | tstop | status | event_num
# Used for: Andersen-Gill, PWP, MCF
make_long_format <- function(df) {
  rows <- vector("list", nrow(df))

  for (i in seq_len(nrow(df))) {
    pid <- df$id[i]
    grp <- as.character(df$group[i])
    fup <- df$followup[i]

    # Event times in days from index operation
    et <- numeric(0)
    if (df$reop1[i] == 1) et <- c(et, days_diff(df$date_operation[i], df$date_reoperation1[i]))
    if (df$reop2[i] == 1) et <- c(et, days_diff(df$date_operation[i], df$date_reoperation2[i]))
    et <- sort(et)

    if (length(et) == 0) {
      # No events: single censored interval [0, followup)
      rows[[i]] <- tibble::tibble(
        id        = pid,
        group     = grp,
        tstart    = 0,
        tstop     = fup,
        status    = 0L,
        event_num = 1L
      )
    } else {
      times <- c(0, et, fup)
      n_int <- length(times) - 1
      rows[[i]] <- tibble::tibble(
        id        = pid,
        group     = grp,
        tstart    = times[1:n_int],
        tstop     = times[2:(n_int + 1)],
        status    = c(rep(1L, length(et)), 0L),
        event_num = seq_len(n_int)
      )
    }
  }

  dplyr::bind_rows(rows) |>
    dplyr::mutate(group = factor(group))
}


In [ ]:
# ---- Build long-format datasets for Part 1 and Part 2 ----
long_df <- make_long_format(patient_df)
long_df_og <- make_long_format(patient_df_og)

cat(sprintf("long_df rows:    %d  |  events: %d\n", nrow(long_df), sum(long_df$status)))
cat(sprintf("long_df_og rows: %d  |  events: %d\n", nrow(long_df_og), sum(long_df_og$status)))


# FULL FOLLOW UP

## 1. KAPLAN-MEIER + LOG-RANK  (time to first re-operation)

### Все группы

In [ ]:
km_fit <- survfit(Surv(time_first, event_first) ~ group, data = first_reop_df)
print(km_fit)

# Log-rank test
logrank <- survdiff(Surv(time_first, event_first) ~ group, data = first_reop_df)
cat("\nLog-rank test:\n")
print(logrank)

# P-value
p_logrank <- pchisq(logrank$chisq, df = length(logrank$n) - 1, lower.tail = FALSE)
cat(sprintf("Log-rank p-value: %.4f\n", p_logrank))

# Plot
km_plot <- ggsurvplot(
  km_fit,
  data           = first_reop_df,
  pval           = TRUE,
  conf.int       = TRUE,
  risk.table     = FALSE,
  palette        = group_palette,
  xlab           = "годы",
  ylab           = "вероятность повторной операции",
  legend.labs    = group_levels,
  break.time.by  = six_months_days, # tick every 0.5 years
  xscale         = 365.25 # convert day values to year labels
)

print(km_plot)


In [ ]:
# Survival estimates at every 6 months (half-year intervals)
half_year_times <- seq(six_months_days,
    max(first_reop_df$time_first, na.rm = TRUE),
    by = six_months_days
)
km_summary <- summary(km_fit, times = half_year_times, extend = TRUE)
print(km_summary)


### ОГ1+ОГ2 vs КГ

In [ ]:
km_fit_og <- survfit(Surv(time_first, event_first) ~ group, data = first_reop_df_og)
print(km_fit_og)

logrank_og <- survdiff(Surv(time_first, event_first) ~ group, data = first_reop_df_og)
cat("\nLog-rank test:\n")
print(logrank_og)

p_logrank_og <- pchisq(logrank_og$chisq, df = length(logrank_og$n) - 1, lower.tail = FALSE)
cat(sprintf("Log-rank p-value: %.4f\n", p_logrank_og))

km_plot_og <- ggsurvplot(
  km_fit_og,
  data           = first_reop_df_og,
  pval           = TRUE,
  conf.int       = TRUE,
  risk.table     = FALSE,
  palette        = group_palette_og,
  xlab           = "годы",
  ylab           = "вероятность повторной операции",
  legend.labs    = group_levels_og,
  break.time.by  = six_months_days,
  xscale         = 365.25
)
print(km_plot_og)

half_year_times_og <- seq(six_months_days,
  max(first_reop_df_og$time_first, na.rm = TRUE),
  by = six_months_days
)
print(summary(km_fit_og, times = half_year_times_og, extend = TRUE))


## 2. COX PROPORTIONAL HAZARDS  (time to first re-operation)

### 2.1  Unadjusted — group only


#### Все группы

In [ ]:
cox_fit <- coxph(Surv(time_first, event_first) ~ group, data = first_reop_df)
cat("\nCox PH model summary (group only):\n")
print(summary(cox_fit))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test <- cox.zph(cox_fit)
print(ph_test)
# p > 0.05 → PH assumption not violated
plot(ph_test)


#### ОГ1+ОГ2 vs КГ

In [ ]:
cox_fit_og <- coxph(Surv(time_first, event_first) ~ group, data = first_reop_df_og)
cat("\nCox PH model summary — ОГ combined (group only):\n")
print(summary(cox_fit_og))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption:\n")
ph_test <- cox.zph(cox_fit_og)
print(ph_test)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test)


### 2.2  Adjusted — group + оперированные пазухи (sinuses)


#### Все группы

In [ ]:
cox_fit_adj <- coxph(Surv(time_first, event_first) ~ group + sinuses, data = first_reop_df)
cat("\nCox PH model summary (group + sinuses):\n")
print(summary(cox_fit_adj))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test_adj <- cox.zph(cox_fit_adj)
print(ph_test_adj)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test_adj)


### ОГ1+ОГ2 vs КГ

In [ ]:
cox_fit_adj_og <- coxph(Surv(time_first, event_first) ~ group + sinuses, data = first_reop_df_og)
cat("\nCox PH model summary — ОГ combined (group + sinuses):\n")
print(summary(cox_fit_adj_og))


In [ ]:
# Plot Schoenfeld residuals for PH test (OG combined)
cat("\nTest of PH assumption:\n")
ph_test_adj_og <- cox.zph(cox_fit_adj_og)
print(ph_test_adj_og)
# Plot Schoenfeld residuals for PH test (OG combined)
plot(ph_test_adj_og)


## 3. ANDERSEN-GILL MODEL  (all re-operations, independent increments)

### Все группы

In [ ]:
# Assumes each event interval is independent (like multiple independent subjects)
# Uses (tstart, tstop] counting process notation

ag_fit <- coxph(
  Surv(tstart, tstop, status) ~ group + cluster(id),
  data   = long_df,
  method = "efron"
)
cat("\nAndersen-Gill model summary:\n")
print(summary(ag_fit))
cat("  HR > 1 means Group B has HIGHER re-op rate than Group A\n")
cat("  HR < 1 means Group B has LOWER  re-op rate (protective)\n")


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test_adj <- cox.zph(ag_fit)
print(ph_test_adj)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test_adj)


### ОГ1+ОГ2 vs КГ

In [ ]:
ag_fit_og <- coxph(
  Surv(tstart, tstop, status) ~ group + cluster(id),
  data   = long_df_og,
  method = "efron"
)
cat("\nAndersen-Gill model summary — ОГ combined:\n")
print(summary(ag_fit_og))


In [ ]:
# Plot Schoenfeld residuals for PH test (OG combined)
cat("\nTest of PH assumption:\n")
ph_test_adj_og <- cox.zph(ag_fit_og)
print(ph_test_adj_og)
# Plot Schoenfeld residuals for PH test (OG combined)
plot(ph_test_adj_og)


## 4. PRENTICE-WILLIAMS-PETERSON (PWP) MODEL  (ordered recurring events)

### Все группы

In [ ]:
# Conditions on prior event â€” models k-th re-operation separately.
# stratify by event number so each stratum gets its own baseline hazard.

# Limit to events 1-3 (small strata beyond that)
pwp_df <- long_df |>
  filter(event_num <= 3) |>
  dplyr::mutate(strata = factor(event_num))

pwp_fit <- coxph(
  Surv(tstart, tstop, status) ~ group + strata(strata) + cluster(id),
  data   = pwp_df,
  method = "efron"
)
cat("\nPWP model summary:\n")
print(summary(pwp_fit))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test_adj <- cox.zph(pwp_fit)
print(ph_test_adj)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test_adj)


In [ ]:
# Per-stratum (per event number) analysis
cat("\nPer-event-number HRs:\n")
for (k in 1:3) {
  df_k <- long_df |> filter(event_num == k)
  if (sum(df_k$status) < 5) next # skip if too few events
  fit_k <- coxph(Surv(tstart, tstop, status) ~ group + cluster(id), data = df_k)
  s <- summary(fit_k)
  hr <- round(s$conf.int[1, 1], 3)
  lo <- round(s$conf.int[1, 3], 3)
  hi <- round(s$conf.int[1, 4], 3)
  pv <- round(s$coefficients[1, 5], 4)
  cat(sprintf(
    "  Event %d: HR = %.3f (95%% CI %.3fâ€“%.3f), p = %.4f\n",
    k, hr, lo, hi, pv
  ))
}


### ОГ1+ОГ2 vs КГ

In [ ]:
pwp_df_og <- long_df_og |>
  filter(event_num <= 3) |>
  dplyr::mutate(strata = factor(event_num))

pwp_fit_og <- coxph(
  Surv(tstart, tstop, status) ~ group + strata(strata) + cluster(id),
  data   = pwp_df_og,
  method = "efron"
)
cat("\nPWP model summary — ОГ combined:\n")
print(summary(pwp_fit_og))


In [ ]:
# Plot Schoenfeld residuals for PH test (OG combined)
cat("\nTest of PH assumption:\n")
ph_test_adj_og <- cox.zph(pwp_fit_og)
print(ph_test_adj_og)
# Plot Schoenfeld residuals for PH test (OG combined)
plot(ph_test_adj_og)


In [ ]:
cat("\nPer-event-number HRs — ОГ combined:\n")
for (k in 1:3) {
  df_k <- long_df_og |> filter(event_num == k)
  if (sum(df_k$status) < 5) next
  fit_k <- coxph(Surv(tstart, tstop, status) ~ group + cluster(id), data = df_k)
  s <- summary(fit_k)
  hr <- round(s$conf.int[1, 1], 3)
  lo <- round(s$conf.int[1, 3], 3)
  hi <- round(s$conf.int[1, 4], 3)
  pv <- round(s$coefficients[1, 5], 4)
  cat(sprintf("  Event %d: HR = %.3f (95%% CI %.3f\u2013%.3f), p = %.4f\n", k, hr, lo, hi, pv))
}


## 5. NEGATIVE BINOMIAL REGRESSION  (total re-operation count)

### Все группы

In [ ]:
# offset(log(followup)) accounts for different observation lengths

nb_fit <- glm.nb(
  n_reops ~ group + offset(log(followup)),
  data = patient_df
)
cat("\nNegative Binomial model summary:\n")
print(summary(nb_fit))

# Rate ratio with CIs
rr <- exp(coef(nb_fit))
rr_ci <- exp(confint(nb_fit))
cat("\nRate Ratios (relative to Group A):\n")
print(cbind(RR = round(rr, 3), round(rr_ci, 3)))

cat(sprintf(
  "\nGroup B has %.1f%% %s re-operations per unit time compared to Group A\n",
  abs(1 - rr["groupB"]) * 100,
  ifelse(rr["groupB"] < 1, "fewer", "more")
))


### ОГ1+ОГ2 vs КГ

In [ ]:
nb_fit_og <- glm.nb(
  n_reops ~ group + offset(log(followup)),
  data = patient_df_og
)
cat("\nNegative Binomial model summary — ОГ combined:\n")
print(summary(nb_fit_og))

rr_og <- exp(coef(nb_fit_og))
rr_ci_og <- exp(confint(nb_fit_og))
cat("\nRate Ratios (ОГ vs КГ):\n")
print(cbind(RR = round(rr_og, 3), round(rr_ci_og, 3)))


## 6. MEAN CUMULATIVE FUNCTION (MCF)

### Все группы

In [ ]:
# Uses the Nelson-Aalen estimator applied to recurring events.
# survfit() with id= argument computes the MCF.

mcf_fit <- survfit(
  Surv(tstart, tstop, status) ~ group,
  data = long_df,
  id   = id # identifies subject; cumhaz gives the MCF for recurring events
)

# Extract MCF data for all groups dynamically
mcf_data <- bind_rows(lapply(seq_along(group_levels), function(i) {
  tibble(
    time  = mcf_fit[i]$time,
    mcf   = mcf_fit[i]$cumhaz,
    lower = mcf_fit[i]$cumhaz - 1.96 * mcf_fit[i]$std.err,
    upper = mcf_fit[i]$cumhaz + 1.96 * mcf_fit[i]$std.err,
    group = group_levels[i]
  )
}))

mcf_plot <- ggplot(mcf_data, aes(x = time, y = mcf, color = group, fill = group)) +
  geom_step(linewidth = 1) +
  geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.15, color = NA) +
  scale_color_manual(values = group_palette) +
  scale_fill_manual(values = group_palette) +
  scale_x_continuous(
    limits       = c(0, NA),
    breaks       = seq(0, max(mcf_data$time), by = six_months_days),
    labels       = function(x) round(x / 365.25, 1),
    minor_breaks = NULL,
    expand       = expansion(mult = c(0, 0.02))
  ) +
  labs(
    # title    = "Среднее значение кумулятивной функции (MCF) повторных операций",
    # subtitle = "Ожидаемое кумулятивное количество повторных операций на пациента с течением времени",
    x        = "годы",
    y        = "Среднее кумулятивное количество повторных операций",
    color    = "Группа",
    fill     = "Группа"
  ) +
  theme_bw(base_size = 16) +
  theme(legend.position = "top")

print(mcf_plot)


### ОГ1+ОГ2 vs КГ

In [ ]:
mcf_fit_og <- survfit(
  Surv(tstart, tstop, status) ~ group,
  data = long_df_og,
  id   = id
)

mcf_data_og <- bind_rows(lapply(seq_along(group_levels_og), function(i) {
  tibble(
    time  = mcf_fit_og[i]$time,
    mcf   = mcf_fit_og[i]$cumhaz,
    lower = mcf_fit_og[i]$cumhaz - 1.96 * mcf_fit_og[i]$std.err,
    upper = mcf_fit_og[i]$cumhaz + 1.96 * mcf_fit_og[i]$std.err,
    group = group_levels_og[i]
  )
}))

mcf_plot_og <- ggplot(mcf_data_og, aes(x = time, y = mcf, color = group, fill = group)) +
  geom_step(linewidth = 1) +
  geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.15, color = NA) +
  scale_color_manual(values = group_palette_og) +
  scale_fill_manual(values = group_palette_og) +
  scale_x_continuous(
    limits       = c(0, NA),
    breaks       = seq(0, max(mcf_data_og$time), by = six_months_days),
    labels       = function(x) round(x / 365.25, 1),
    minor_breaks = NULL,
    expand       = expansion(mult = c(0, 0.02))
  ) +
  labs(
    x     = "годы",
    y     = "Среднее кумулятивное количество повторных операций",
    color = "Группа",
    fill  = "Группа"
  ) +
  theme_bw(base_size = 16) +
  theme(legend.position = "top")

print(mcf_plot_og)


## FOREST PLOT — all effect estimates in one figure

### Все группы

In [ ]:
# All models report a ratio (HR or RR) relative to the reference group.
#   ratio < 1  →  group has FEWER / LATER re-operations  (better)
#   ratio > 1  →  group has MORE  / EARLIER re-operations (worse)

# ---- Helper: extract HR/CI/p from a coxph or glm model ----
extract_hr <- function(fit, coef_row = 1) {
  s <- summary(fit)
  ci <- exp(confint(fit))
  if ("conf.int" %in% names(s)) {
    hr <- s$conf.int[coef_row, 1]
    lo <- s$conf.int[coef_row, 3]
    hi <- s$conf.int[coef_row, 4]
    pv <- s$coefficients[coef_row, ncol(s$coefficients)]
  } else {
    hr <- exp(s$coefficients[coef_row, 1])
    lo <- ci[coef_row, 1]
    hi <- ci[coef_row, 2]
    pv <- s$coefficients[coef_row, 4]
  }
  list(est = hr, lo = lo, hi = hi, p = pv)
}

# ---- Helper: extract NB rate ratio (first group coefficient) ----
extract_nb <- function(nb_model) {
  s <- summary(nb_model)
  ci <- exp(confint(nb_model))
  grp_rows <- grep("^group", rownames(s$coefficients))
  if (length(grp_rows) == 0) stop("No 'group' coefficient found in NB model")
  rname <- rownames(s$coefficients)[grp_rows[1]]
  list(
    est   = exp(s$coefficients[rname, 1]),
    lo    = ci[rname, 1],
    hi    = ci[rname, 2],
    p     = s$coefficients[rname, 4],
    label = sub("^group", "", rname)
  )
}

# ---- Helper: build a forest plot ----
make_forest_plot <- function(cox_e, ag_e, pwp_e, nb_e, title_str) {
  model_labels <- c(
    "Cox PH\n(time to 1st re-op)",
    "Andersen-Gill\n(all re-ops)",
    "PWP\n(ordered re-ops)",
    "Neg. Binomial\n(total count)"
  )
  fdf <- tibble::tibble(
    Model    = factor(model_labels, levels = rev(model_labels)),
    Estimate = c(cox_e$est, ag_e$est, pwp_e$est, nb_e$est),
    Lower    = c(cox_e$lo, ag_e$lo, pwp_e$lo, nb_e$lo),
    Upper    = c(cox_e$hi, ag_e$hi, pwp_e$hi, nb_e$hi),
    P        = c(cox_e$p, ag_e$p, pwp_e$p, nb_e$p)
  ) |>
    dplyr::mutate(
      label = sprintf(
        "%.2f (%.2f\u2013%.2f)\np=%s",
        Estimate, Lower, Upper,
        ifelse(P < 0.001, "<0.001", sprintf("%.3f", P))
      ),
      sig = ifelse(P < 0.05, "Significant", "Non-significant"),
      colour = ifelse(Estimate < 1, "#2ECC71", "#E74C3C")
    )

  ggplot2::ggplot(fdf, aes(x = Estimate, y = Model, xmin = Lower, xmax = Upper)) +
    ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
    ggplot2::geom_errorbarh(aes(colour = colour), height = 0.2, linewidth = 0.8) +
    ggplot2::geom_point(aes(colour = colour, shape = sig), size = 3.5) +
    ggplot2::geom_text(aes(x = max(Upper) * 1.05, label = label),
      hjust = 0, size = 3, lineheight = 0.9
    ) +
    ggplot2::scale_colour_identity() +
    ggplot2::scale_shape_manual(values = c("Significant" = 16, "Non-significant" = 1)) +
    ggplot2::scale_x_continuous(
      expand = ggplot2::expansion(mult = c(0.05, 0.45)),
      trans  = "log10"
    ) +
    labs(
      title    = title_str,
      subtitle = "Ratio < 1 (green): fewer/later re-operations (BETTER)\nRatio > 1 (red): more/earlier re-operations (WORSE)",
      x        = "Hazard Ratio / Rate Ratio (log scale)",
      y        = NULL,
      shape    = NULL
    ) +
    ggplot2::theme_bw(base_size = 16) +
    theme(
      legend.position = "bottom",
      panel.grid.minor = element_blank()
    )
}

# ---- Все группы: extract estimates ----
cox_e <- extract_hr(cox_fit)
ag_e <- extract_hr(ag_fit)
pwp_e <- extract_hr(pwp_fit)
nb_e <- extract_nb(nb_fit)

forest_plot <- make_forest_plot(
  cox_e, ag_e, pwp_e, nb_e,
  title_str = sprintf(
    "Все группы (%s vs reference) \u2014 Effect Estimates",
    nb_e$label
  )
)
print(forest_plot)


### ОГ1+ОГ2 vs КГ

In [ ]:
# ---- ОГ combined: extract estimates ----
cox_e_og <- extract_hr(cox_fit_og)
ag_e_og <- extract_hr(ag_fit_og)
pwp_e_og <- extract_hr(pwp_fit_og)
nb_e_og <- extract_nb(nb_fit_og)

forest_plot_og <- make_forest_plot(
  cox_e_og, ag_e_og, pwp_e_og, nb_e_og,
  title_str = sprintf(
    "%s vs \u041a\u0413 \u2014 Effect Estimates (\u041e\u04131+\u041e\u04132 combined)",
    nb_e_og$label
  )
)
print(forest_plot_og)


## GROUP COMPARISON SUMMARY 

### Все группы

In [ ]:
# ---- Helper: interpret a single ratio ----
interpret <- function(label, est, lo, hi, p, ref, grp, unit = "повторные операции") {
  direction <- if (est < 1) "МЕНЬШЕ" else "БОЛЬШЕ"
  verdict <- if (est < 1) "ЛУЧШЕ" else "ХУЖЕ"
  sig_str <- if (p < 0.05) {
    sprintf("(p=%.4f, статистически значимо)", p)
  } else {
    sprintf("(p=%.4f, статистически незначимо)", p)
  }
  pct <- abs(1 - est) * 100
  cat(sprintf(
    "  %-35s  HR/RR = %.2f (95%% ДИ %.2f\u2013%.2f)\n    \u2192 %s имеет %.1f%% %s %s чем %s  [%s]  %s\n\n",
    label, est, lo, hi, grp, pct, direction, unit, ref, verdict, sig_str
  ))
}

# ---- Helper: print verdict block ----
print_verdict <- function(ests_list) {
  all_ests <- sapply(ests_list, `[[`, "est")
  any_sig <- any(sapply(ests_list, `[[`, "p") < 0.05)
  all_same <- all(all_ests < 1) || all(all_ests > 1)
  cat("--- ОБЩЕЕ ЗАКЛЮЧЕНИЕ ---\n")
  if (all_same && any_sig) {
    dir <- if (all(all_ests < 1)) "МЕНЬШЕ" else "БОЛЬШЕ"
    cat(sprintf("  Все модели последовательно показывают одно направление (%s повторные операции)\n  и как минимум один результат статистически значим.\n", dir))
  } else if (all_same && !any_sig) {
    dir <- if (all(all_ests < 1)) "МЕНЬШЕ" else "БОЛЬШЕ"
    cat(sprintf("  Все модели указывают в одном направлении (%s повторные операции)\n  но ни один результат не достигает статистической значимости.\n  Рассмотрите возможность увеличения размера выборки.\n", dir))
  } else {
    cat("  Результаты непоследовательны между моделями.\n")
    cat("  Проверьте предположения моделей и качество данных.\n")
  }
}

# ---- Все группы ----
km_med <- surv_median(km_fit)
cat("--- Время до 1-й повторной операции (медианы Kaplan-Meier) ---\n")
for (i in seq_len(nrow(km_med))) {
  cat(sprintf(
    "  %s:  медиана = %.1f дней  (95%% ДИ %.1f\u2013%.1f)\n",
    km_med$strata[i], km_med$median[i],
    km_med$lower[i], km_med$upper[i]
  ))
}
p_lr <- pchisq(logrank$chisq, df = length(logrank$n) - 1, lower.tail = FALSE)
cat(sprintf(
  "  Log-rank p-value: %.4f  %s\n\n",
  p_lr,
  ifelse(p_lr < 0.05,
    "\u2192 Группы статистически значимо различаются.",
    "\u2192 Статистически значимых различий нет."
  )
))

ref_all <- levels(first_reop_df$group)[1]
grp_all <- nb_e$label
cat(sprintf("--- Оценка эффекта (опорная группа: %s) ---\n", ref_all))
interpret("Cox PH (время до 1-й повторной операции)", cox_e$est, cox_e$lo, cox_e$hi, cox_e$p, ref_all, grp_all)
interpret("Andersen-Gill (все повторные операции)", ag_e$est, ag_e$lo, ag_e$hi, ag_e$p, ref_all, grp_all)
interpret("PWP (упорядоченные повторные операции)", pwp_e$est, pwp_e$lo, pwp_e$hi, pwp_e$p, ref_all, grp_all)
interpret("Neg. Binomial (общее количество)", nb_e$est, nb_e$lo, nb_e$hi, nb_e$p, ref_all, grp_all,
  unit = "повторные операции за единицу времени"
)
print_verdict(list(cox_e, ag_e, pwp_e, nb_e))


### ОГ1+ОГ2 vs КГ

In [ ]:
# ---- ОГ1+ОГ2 vs КГ ----
km_med_og <- surv_median(km_fit_og)
cat("--- Время до 1-й повторной операции (медианы Kaplan-Meier) ---\n")
for (i in seq_len(nrow(km_med_og))) {
  cat(sprintf(
    "  %s:  медиана = %.1f дней  (95%% ДИ %.1f\u2013%.1f)\n",
    km_med_og$strata[i], km_med_og$median[i],
    km_med_og$lower[i], km_med_og$upper[i]
  ))
}
p_lr_og <- pchisq(logrank_og$chisq, df = length(logrank_og$n) - 1, lower.tail = FALSE)
cat(sprintf(
  "  Log-rank p-value: %.4f  %s\n\n",
  p_lr_og,
  ifelse(p_lr_og < 0.05,
    "\u2192 Группы статистически значимо различаются.",
    "\u2192 Статистически значимых различий нет."
  )
))

ref_og <- levels(first_reop_df_og$group)[1]
grp_og <- nb_e_og$label
cat(sprintf("--- Оценка эффекта (опорная группа: %s) ---\n", ref_og))
interpret("Cox PH (время до 1-й повторной операции)", cox_e_og$est, cox_e_og$lo, cox_e_og$hi, cox_e_og$p, ref_og, grp_og)
interpret("Andersen-Gill (все повторные операции)", ag_e_og$est, ag_e_og$lo, ag_e_og$hi, ag_e_og$p, ref_og, grp_og)
interpret("PWP (упорядоченные повторные операции)", pwp_e_og$est, pwp_e_og$lo, pwp_e_og$hi, pwp_e_og$p, ref_og, grp_og)
interpret("Neg. Binomial (общее количество)", nb_e_og$est, nb_e_og$lo, nb_e_og$hi, nb_e_og$p, ref_og, grp_og,
  unit = "повторные операции за единицу времени"
)
print_verdict(list(cox_e_og, ag_e_og, pwp_e_og, nb_e_og))


# FOLLOW UP TIME HARD CUT-OFF: 3 YEARS

## Cut follow-up at 3 years

In [ ]:
# ---- 3-year hard cut-off censoring ----
# All re-operations occurring AFTER this threshold are treated as censored.
cutoff_3y <- 3 * 365.25 # 1095.75 days

# --- first_reop_df / first_reop_df_og ---
# (wide format: one row per patient, time to first re-op)
first_reop_df <- first_reop_df |>
  dplyr::mutate(
    event_first = ifelse(time_first > cutoff_3y, 0L, event_first),
    time_first  = pmin(time_first, cutoff_3y)
  )

first_reop_df_og <- first_reop_df_og |>
  dplyr::mutate(
    event_first = ifelse(time_first > cutoff_3y, 0L, event_first),
    time_first  = pmin(time_first, cutoff_3y)
  )

# --- patient_df / patient_df_og ---
# (counts of re-ops + follow-up time; used in Negative Binomial regression)
censor_patient <- function(df, cut) {
  df |>
    dplyr::mutate(
      reop1_in = reop1 == 1 & !is.na(date_reoperation1) &
        days_diff(date_operation, date_reoperation1) <= cut,
      reop2_in = reop2 == 1 & !is.na(date_reoperation2) &
        days_diff(date_operation, date_reoperation2) <= cut,
      n_reops = as.integer(reop1_in) + as.integer(reop2_in),
      followup = pmin(followup, cut)
    ) |>
    select(-reop1_in, -reop2_in)
}

patient_df <- censor_patient(patient_df, cutoff_3y)
patient_df_og <- censor_patient(patient_df_og, cutoff_3y)

# --- long_df / long_df_og ---
# (counting-process format; used in Andersen-Gill, PWP, MCF)
censor_long <- function(df, cut) {
  df |>
    filter(tstart < cut) |>
    dplyr::mutate(
      status = ifelse(tstop > cut, 0L, status),
      tstop  = pmin(tstop, cut)
    )
}

long_df <- censor_long(long_df, cutoff_3y)
long_df_og <- censor_long(long_df_og, cutoff_3y)

# head(first_reop_df, n=50)


## 1. KAPLAN-MEIER + LOG-RANK  (time to first re-operation)

### Все группы

In [ ]:
km_fit <- survfit(Surv(time_first, event_first) ~ group, data = first_reop_df)
print(km_fit)

# Log-rank test
logrank <- survdiff(Surv(time_first, event_first) ~ group, data = first_reop_df)
cat("\nLog-rank test:\n")
print(logrank)

# P-value
p_logrank <- pchisq(logrank$chisq, df = length(logrank$n) - 1, lower.tail = FALSE)
cat(sprintf("Log-rank p-value: %.4f\n", p_logrank))

# Plot
km_plot <- ggsurvplot(
  km_fit,
  data           = first_reop_df,
  pval           = TRUE,
  conf.int       = TRUE,
  risk.table     = FALSE,
  palette        = group_palette,
  xlab           = "годы",
  ylab           = "вероятность повторной операции",
  legend.labs    = group_levels,
  break.time.by  = six_months_days, # tick every 0.5 years
  xscale         = 365.25 # convert day values to year labels
)

print(km_plot)


In [ ]:
# Survival estimates at every 6 months (half-year intervals)
half_year_times <- seq(six_months_days,
    max(first_reop_df$time_first, na.rm = TRUE),
    by = six_months_days
)
km_summary <- summary(km_fit, times = half_year_times, extend = TRUE)
print(km_summary)


### ОГ1+ОГ2 vs КГ

In [ ]:
km_fit_og <- survfit(Surv(time_first, event_first) ~ group, data = first_reop_df_og)
print(km_fit_og)

logrank_og <- survdiff(Surv(time_first, event_first) ~ group, data = first_reop_df_og)
cat("\nLog-rank test:\n")
print(logrank_og)

p_logrank_og <- pchisq(logrank_og$chisq, df = length(logrank_og$n) - 1, lower.tail = FALSE)
cat(sprintf("Log-rank p-value: %.4f\n", p_logrank_og))

km_plot_og <- ggsurvplot(
  km_fit_og,
  data           = first_reop_df_og,
  pval           = TRUE,
  conf.int       = TRUE,
  risk.table     = FALSE,
  palette        = group_palette_og,
  xlab           = "годы",
  ylab           = "вероятность повторной операции",
  legend.labs    = group_levels_og,
  break.time.by  = six_months_days,
  xscale         = 365.25
)
print(km_plot_og)

half_year_times_og <- seq(six_months_days,
  max(first_reop_df_og$time_first, na.rm = TRUE),
  by = six_months_days
)
print(summary(km_fit_og, times = half_year_times_og, extend = TRUE))


## 2. COX PROPORTIONAL HAZARDS  (time to first re-operation)

### 2.1  Unadjusted group only


#### Все группы

In [ ]:
cox_fit <- coxph(Surv(time_first, event_first) ~ group, data = first_reop_df)
cat("\nCox PH model summary (group only):\n")
print(summary(cox_fit))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption:\n")
ph_test <- cox.zph(cox_fit_og)
print(ph_test)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test)


#### ОГ1+ОГ2 vs КГ

In [ ]:
cox_fit_og <- coxph(Surv(time_first, event_first) ~ group, data = first_reop_df_og)
cat("\nCox PH model summary — ОГ combined (group only):\n")
print(summary(cox_fit_og))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption:\n")
ph_test <- cox.zph(cox_fit_og)
print(ph_test)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test)


### 2.2  Adjusted — group + оперированные пазухи (sinuses)


#### Все группы

In [ ]:
cox_fit_adj <- coxph(Surv(time_first, event_first) ~ group + sinuses, data = first_reop_df)
cat("\nCox PH model summary (group + sinuses):\n")
print(summary(cox_fit_adj))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test_adj <- cox.zph(cox_fit_adj)
print(ph_test_adj)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test_adj)


### ОГ1+ОГ2 vs КГ

In [ ]:
cox_fit_adj_og <- coxph(Surv(time_first, event_first) ~ group + sinuses, data = first_reop_df_og)
cat("\nCox PH model summary — ОГ combined (group + sinuses):\n")
print(summary(cox_fit_adj_og))


In [ ]:
# Plot Schoenfeld residuals for PH test (OG combined)
cat("\nTest of PH assumption:\n")
ph_test_adj_og <- cox.zph(cox_fit_adj_og)
print(ph_test_adj_og)
# Plot Schoenfeld residuals for PH test (OG combined)
plot(ph_test_adj_og)


## 3. ANDERSEN-GILL MODEL  (all re-operations, independent increments)

### Все группы

In [ ]:
# Assumes each event interval is independent (like multiple independent subjects)
# Uses (tstart, tstop] counting process notation

ag_fit <- coxph(
  Surv(tstart, tstop, status) ~ group + cluster(id),
  data   = long_df,
  method = "efron"
)
cat("\nAndersen-Gill model summary:\n")
print(summary(ag_fit))
cat("  HR > 1 means Group B has HIGHER re-op rate than Group A\n")
cat("  HR < 1 means Group B has LOWER  re-op rate (protective)\n")


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test_adj <- cox.zph(ag_fit)
print(ph_test_adj)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test_adj)


### ОГ1+ОГ2 vs КГ

In [ ]:
ag_fit_og <- coxph(
  Surv(tstart, tstop, status) ~ group + cluster(id),
  data   = long_df_og,
  method = "efron"
)
cat("\nAndersen-Gill model summary — ОГ combined:\n")
print(summary(ag_fit_og))


In [ ]:
# Plot Schoenfeld residuals for PH test (OG combined)
cat("\nTest of PH assumption:\n")
ph_test_adj_og <- cox.zph(ag_fit_og)
print(ph_test_adj_og)
# Plot Schoenfeld residuals for PH test (OG combined)
plot(ph_test_adj_og)


## 4. PRENTICE-WILLIAMS-PETERSON (PWP) MODEL  (ordered recurring events)

### Все группы

In [ ]:
# Conditions on prior event â€” models k-th re-operation separately.
# stratify by event number so each stratum gets its own baseline hazard.

# Limit to events 1-3 (small strata beyond that)
pwp_df <- long_df |>
  filter(event_num <= 3) |>
  dplyr::mutate(strata = factor(event_num))

pwp_fit <- coxph(
  Surv(tstart, tstop, status) ~ group + strata(strata) + cluster(id),
  data   = pwp_df,
  method = "efron"
)
cat("\nPWP model summary:\n")
print(summary(pwp_fit))


In [ ]:
# Test PH assumption (Schoenfeld residuals)
cat("\nTest of PH assumption (Schoenfeld residuals):\n")
ph_test_adj <- cox.zph(pwp_fit)
print(ph_test_adj)
# p > 0.05 → PH assumption not violated

# Plot Schoenfeld residuals for PH test
plot(ph_test_adj)


In [ ]:
# Per-stratum (per event number) analysis
cat("\nPer-event-number HRs:\n")
for (k in 1:3) {
  df_k <- long_df |> filter(event_num == k)
  if (sum(df_k$status) < 5) next # skip if too few events
  fit_k <- coxph(Surv(tstart, tstop, status) ~ group + cluster(id), data = df_k)
  s <- summary(fit_k)
  hr <- round(s$conf.int[1, 1], 3)
  lo <- round(s$conf.int[1, 3], 3)
  hi <- round(s$conf.int[1, 4], 3)
  pv <- round(s$coefficients[1, 5], 4)
  cat(sprintf(
    "  Event %d: HR = %.3f (95%% CI %.3fâ€“%.3f), p = %.4f\n",
    k, hr, lo, hi, pv
  ))
}


### ОГ1+ОГ2 vs КГ

In [ ]:
pwp_df_og <- long_df_og |>
  filter(event_num <= 3) |>
  dplyr::mutate(strata = factor(event_num))

pwp_fit_og <- coxph(
  Surv(tstart, tstop, status) ~ group + strata(strata) + cluster(id),
  data   = pwp_df_og,
  method = "efron"
)
cat("\nPWP model summary — ОГ combined:\n")
print(summary(pwp_fit_og))


In [ ]:
# Plot Schoenfeld residuals for PH test (OG combined)
cat("\nTest of PH assumption:\n")
ph_test_adj_og <- cox.zph(pwp_fit_og)
print(ph_test_adj_og)
# Plot Schoenfeld residuals for PH test (OG combined)
plot(ph_test_adj_og)


In [ ]:
cat("\nPer-event-number HRs — ОГ combined:\n")
for (k in 1:3) {
  df_k <- long_df_og |> filter(event_num == k)
  if (sum(df_k$status) < 5) next
  fit_k <- coxph(Surv(tstart, tstop, status) ~ group + cluster(id), data = df_k)
  s <- summary(fit_k)
  hr <- round(s$conf.int[1, 1], 3)
  lo <- round(s$conf.int[1, 3], 3)
  hi <- round(s$conf.int[1, 4], 3)
  pv <- round(s$coefficients[1, 5], 4)
  cat(sprintf("  Event %d: HR = %.3f (95%% CI %.3f\u2013%.3f), p = %.4f\n", k, hr, lo, hi, pv))
}


## 5. NEGATIVE BINOMIAL REGRESSION  (total re-operation count)

### Все группы

In [ ]:
# offset(log(followup)) accounts for different observation lengths

nb_fit <- glm.nb(
  n_reops ~ group + offset(log(followup)),
  data = patient_df
)
cat("\nNegative Binomial model summary:\n")
print(summary(nb_fit))

# Rate ratio with CIs
rr <- exp(coef(nb_fit))
rr_ci <- exp(confint(nb_fit))
cat("\nRate Ratios (relative to Group A):\n")
print(cbind(RR = round(rr, 3), round(rr_ci, 3)))

cat(sprintf(
  "\nGroup B has %.1f%% %s re-operations per unit time compared to Group A\n",
  abs(1 - rr["groupB"]) * 100,
  ifelse(rr["groupB"] < 1, "fewer", "more")
))


### ОГ1+ОГ2 vs КГ

In [ ]:
nb_fit_og <- glm.nb(
  n_reops ~ group + offset(log(followup)),
  data = patient_df_og
)
cat("\nNegative Binomial model summary — ОГ combined:\n")
print(summary(nb_fit_og))

rr_og <- exp(coef(nb_fit_og))
rr_ci_og <- exp(confint(nb_fit_og))
cat("\nRate Ratios (ОГ vs КГ):\n")
print(cbind(RR = round(rr_og, 3), round(rr_ci_og, 3)))


## 6. MEAN CUMULATIVE FUNCTION (MCF)

### Все группы

In [ ]:
# Uses the Nelson-Aalen estimator applied to recurring events.
# survfit() with id= argument computes the MCF.

mcf_fit <- survfit(
  Surv(tstart, tstop, status) ~ group,
  data = long_df,
  id   = id # identifies subject; cumhaz gives the MCF for recurring events
)

# Extract MCF data for all groups dynamically
mcf_data <- bind_rows(lapply(seq_along(group_levels), function(i) {
  tibble(
    time  = mcf_fit[i]$time,
    mcf   = mcf_fit[i]$cumhaz,
    lower = mcf_fit[i]$cumhaz - 1.96 * mcf_fit[i]$std.err,
    upper = mcf_fit[i]$cumhaz + 1.96 * mcf_fit[i]$std.err,
    group = group_levels[i]
  )
}))

mcf_plot <- ggplot(mcf_data, aes(x = time, y = mcf, color = group, fill = group)) +
  geom_step(linewidth = 1) +
  geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.15, color = NA) +
  scale_color_manual(values = group_palette) +
  scale_fill_manual(values = group_palette) +
  scale_x_continuous(
    limits       = c(0, NA),
    breaks       = seq(0, max(mcf_data$time), by = six_months_days),
    labels       = function(x) round(x / 365.25, 1),
    minor_breaks = NULL,
    expand       = expansion(mult = c(0, 0.02))
  ) +
  labs(
    # title    = "Среднее значение кумулятивной функции (MCF) повторных операций",
    # subtitle = "Ожидаемое кумулятивное количество повторных операций на пациента с течением времени",
    x        = "годы",
    y        = "Среднее кумулятивное количество повторных операций",
    color    = "Группа",
    fill     = "Группа"
  ) +
  theme_bw(base_size = 16) +
  theme(legend.position = "top")

print(mcf_plot)


### ОГ1+ОГ2 vs КГ

In [ ]:
mcf_fit_og <- survfit(
  Surv(tstart, tstop, status) ~ group,
  data = long_df_og,
  id   = id
)

mcf_data_og <- bind_rows(lapply(seq_along(group_levels_og), function(i) {
  tibble(
    time  = mcf_fit_og[i]$time,
    mcf   = mcf_fit_og[i]$cumhaz,
    lower = mcf_fit_og[i]$cumhaz - 1.96 * mcf_fit_og[i]$std.err,
    upper = mcf_fit_og[i]$cumhaz + 1.96 * mcf_fit_og[i]$std.err,
    group = group_levels_og[i]
  )
}))

mcf_plot_og <- ggplot(mcf_data_og, aes(x = time, y = mcf, color = group, fill = group)) +
  geom_step(linewidth = 1) +
  geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.15, color = NA) +
  scale_color_manual(values = group_palette_og) +
  scale_fill_manual(values = group_palette_og) +
  scale_x_continuous(
    limits       = c(0, NA),
    breaks       = seq(0, max(mcf_data_og$time), by = six_months_days),
    labels       = function(x) round(x / 365.25, 1),
    minor_breaks = NULL,
    expand       = expansion(mult = c(0, 0.02))
  ) +
  labs(
    x     = "годы",
    y     = "Среднее кумулятивное количество повторных операций",
    color = "Группа",
    fill  = "Группа"
  ) +
  theme_bw(base_size = 16) +
  theme(legend.position = "top")

print(mcf_plot_og)


## FOREST PLOT — all effect estimates in one figure

### Все группы

In [ ]:
# All models report a ratio (HR or RR) relative to the reference group.
#   ratio < 1  →  group has FEWER / LATER re-operations  (better)
#   ratio > 1  →  group has MORE  / EARLIER re-operations (worse)

# ---- Helper: extract HR/CI/p from a coxph or glm model ----
extract_hr <- function(fit, coef_row = 1) {
  s <- summary(fit)
  ci <- exp(confint(fit))
  if ("conf.int" %in% names(s)) {
    hr <- s$conf.int[coef_row, 1]
    lo <- s$conf.int[coef_row, 3]
    hi <- s$conf.int[coef_row, 4]
    pv <- s$coefficients[coef_row, ncol(s$coefficients)]
  } else {
    hr <- exp(s$coefficients[coef_row, 1])
    lo <- ci[coef_row, 1]
    hi <- ci[coef_row, 2]
    pv <- s$coefficients[coef_row, 4]
  }
  list(est = hr, lo = lo, hi = hi, p = pv)
}

# ---- Helper: extract NB rate ratio (first group coefficient) ----
extract_nb <- function(nb_model) {
  s <- summary(nb_model)
  ci <- exp(confint(nb_model))
  grp_rows <- grep("^group", rownames(s$coefficients))
  if (length(grp_rows) == 0) stop("No 'group' coefficient found in NB model")
  rname <- rownames(s$coefficients)[grp_rows[1]]
  list(
    est   = exp(s$coefficients[rname, 1]),
    lo    = ci[rname, 1],
    hi    = ci[rname, 2],
    p     = s$coefficients[rname, 4],
    label = sub("^group", "", rname)
  )
}

# ---- Helper: build a forest plot ----
make_forest_plot <- function(cox_e, ag_e, pwp_e, nb_e, title_str) {
  model_labels <- c(
    "Cox PH\n(время до 1-й повторной операции)",
    "Andersen-Gill\n(все повторные операции)",
    "PWP\n(упорядоченные повторные операции)",
    "Neg. Binomial\n(общее количество)"
  )
  fdf <- tibble(
    Model    = factor(model_labels, levels = rev(model_labels)),
    Estimate = c(cox_e$est, ag_e$est, pwp_e$est, nb_e$est),
    Lower    = c(cox_e$lo, ag_e$lo, pwp_e$lo, nb_e$lo),
    Upper    = c(cox_e$hi, ag_e$hi, pwp_e$hi, nb_e$hi),
    P        = c(cox_e$p, ag_e$p, pwp_e$p, nb_e$p)
  ) |>
    dplyr::mutate(
      label = sprintf(
        "%.2f (%.2f\u2013%.2f)\np=%s",
        Estimate, Lower, Upper,
        ifelse(P < 0.001, "<0.001", sprintf("%.3f", P))
      ),
      sig = ifelse(P < 0.05, "Significant", "Non-significant"),
      colour = ifelse(Estimate < 1, "#2ECC71", "#E74C3C")
    )

  ggplot2::ggplot(fdf, aes(x = Estimate, y = Model, xmin = Lower, xmax = Upper)) +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
    geom_errorbarh(aes(colour = colour), height = 0.2, linewidth = 0.8) +
    geom_point(aes(colour = colour, shape = sig), size = 3.5) +
    geom_text(aes(x = max(Upper) * 1.05, label = label),
      hjust = 0, size = 3, lineheight = 0.9
    ) +
    scale_colour_identity() +
    scale_shape_manual(values = c("Significant" = 16, "Non-significant" = 1)) +
    scale_x_continuous(
      expand = expansion(mult = c(0.05, 0.45)),
      trans  = "log10"
    ) +
    labs(
      title    = title_str,
      subtitle = "Отношение < 1 (зелёный цвет): меньше/позднее повторные операции (ЛУЧШЕ)\nОтношение > 1 (красный цвет): больше/раннее повторные операции (ХУЖЕ)",
      x        = "Отношение рисков / Отношение долей (логарифмическая шкала)",
      y        = NULL,
      shape    = NULL
    ) +
    theme_bw(base_size = 16) +
    theme(
      legend.position = "bottom",
      panel.grid.minor = element_blank()
    )
}

# ---- Все группы: extract estimates ----
cox_e <- extract_hr(cox_fit)
ag_e <- extract_hr(ag_fit)
pwp_e <- extract_hr(pwp_fit)
nb_e <- extract_nb(nb_fit)

forest_plot <- make_forest_plot(
  cox_e, ag_e, pwp_e, nb_e,
  title_str = sprintf(
    "Все группы (%s vs опорная группа) \u2014 Оценка эффекта (%s vs опорная группа) \u2014 Effect Estimates",
    nb_e$label, nb_e$label
  )
)
print(forest_plot)


### ОГ1+ОГ2 vs КГ

In [ ]:
# ---- ОГ combined: extract estimates ----
cox_e_og <- extract_hr(cox_fit_og)
ag_e_og <- extract_hr(ag_fit_og)
pwp_e_og <- extract_hr(pwp_fit_og)
nb_e_og <- extract_nb(nb_fit_og)

forest_plot_og <- make_forest_plot(
  cox_e_og, ag_e_og, pwp_e_og, nb_e_og,
  title_str = sprintf(
    "%s vs \u041a\u0413 \u2014 Оценнка эффекта (ОГ1+ОГ2)",
    nb_e_og$label
  )
)
print(forest_plot_og)


## GROUP COMPARISON SUMMARY 

### Все группы

In [ ]:
# ---- Helper: interpret a single ratio ----
interpret <- function(label, est, lo, hi, p, ref, grp, unit = "повторные операции") {
  direction <- if (est < 1) "МЕНЬШЕ" else "БОЛЬШЕ"
  verdict <- if (est < 1) "ЛУЧШЕ" else "ХУЖЕ"
  sig_str <- if (p < 0.05) {
    sprintf("(p=%.4f, статистически значимо)", p)
  } else {
    sprintf("(p=%.4f, статистически незначимо)", p)
  }
  pct <- abs(1 - est) * 100
  cat(sprintf(
    "  %-35s  HR/RR = %.2f (95%% ДИ %.2f\u2013%.2f)\n    \u2192 %s имеет %.1f%% %s %s чем %s  [%s]  %s\n\n",
    label, est, lo, hi, grp, pct, direction, unit, ref, verdict, sig_str
  ))
}

# ---- Helper: print verdict block ----
print_verdict <- function(ests_list) {
  all_ests <- sapply(ests_list, `[[`, "est")
  any_sig <- any(sapply(ests_list, `[[`, "p") < 0.05)
  all_same <- all(all_ests < 1) || all(all_ests > 1)
  cat("--- ОБЩЕЕ ЗАКЛЮЧЕНИЕ ---\n")
  if (all_same && any_sig) {
    dir <- if (all(all_ests < 1)) "МЕНЬШЕ" else "БОЛЬШЕ"
    cat(sprintf("  Все модели последовательно показывают одно направление (%s повторные операции)\n  и как минимум один результат статистически значим.\n", dir))
  } else if (all_same && !any_sig) {
    dir <- if (all(all_ests < 1)) "МЕНЬШЕ" else "БОЛЬШЕ"
    cat(sprintf("  Все модели указывают в одном направлении (%s повторные операции)\n  но ни один результат не достигает статистической значимости.\n  Рассмотрите возможность увеличения размера выборки.\n", dir))
  } else {
    cat("  Результаты непоследовательны между моделями.\n")
    cat("  Проверьте предположения моделей и качество данных.\n")
  }
}

# ---- Все группы ----
km_med <- surv_median(km_fit)
cat("--- Время до 1-й повторной операции (медианы Kaplan-Meier) ---\n")
for (i in seq_len(nrow(km_med))) {
  cat(sprintf(
    "  %s:  медиана = %.1f дней  (95%% ДИ %.1f\u2013%.1f)\n",
    km_med$strata[i], km_med$median[i],
    km_med$lower[i], km_med$upper[i]
  ))
}
p_lr <- pchisq(logrank$chisq, df = length(logrank$n) - 1, lower.tail = FALSE)
cat(sprintf(
  "  Log-rank p-value: %.4f  %s\n\n",
  p_lr,
  ifelse(p_lr < 0.05,
    "\u2192 Группы статистически значимо различаются.",
    "\u2192 Статистически значимых различий нет."
  )
))

ref_all <- levels(first_reop_df$group)[1]
grp_all <- nb_e$label
cat(sprintf("--- Оценка эффекта (опорная группа: %s) ---\n", ref_all))
interpret("Cox PH (время до 1-й повторной операции)", cox_e$est, cox_e$lo, cox_e$hi, cox_e$p, ref_all, grp_all)
interpret("Andersen-Gill (все повторные операции)", ag_e$est, ag_e$lo, ag_e$hi, ag_e$p, ref_all, grp_all)
interpret("PWP (упорядоченные повторные операции)", pwp_e$est, pwp_e$lo, pwp_e$hi, pwp_e$p, ref_all, grp_all)
interpret("Neg. Binomial (общее количество)", nb_e$est, nb_e$lo, nb_e$hi, nb_e$p, ref_all, grp_all,
  unit = "повторные операции за единицу времени"
)
print_verdict(list(cox_e, ag_e, pwp_e, nb_e))


### ОГ1+ОГ2 vs КГ

In [ ]:
# ---- ОГ1+ОГ2 vs КГ ----
km_med_og <- surv_median(km_fit_og)
cat("--- Время до 1-й повторной операции (медианы Kaplan-Meier) ---\n")
for (i in seq_len(nrow(km_med_og))) {
  cat(sprintf(
    "  %s:  медиана = %.1f дней  (95%% ДИ %.1f\u2013%.1f)\n",
    km_med_og$strata[i], km_med_og$median[i],
    km_med_og$lower[i], km_med_og$upper[i]
  ))
}
p_lr_og <- pchisq(logrank_og$chisq, df = length(logrank_og$n) - 1, lower.tail = FALSE)
cat(sprintf(
  "  Log-rank p-value: %.4f  %s\n\n",
  p_lr_og,
  ifelse(p_lr_og < 0.05,
    "\u2192 Группы статистически значимо различаются.",
    "\u2192 Статистически значимых различий нет."
  )
))

ref_og <- levels(first_reop_df_og$group)[1]
grp_og <- nb_e_og$label
cat(sprintf("--- Оценка эффекта (опорная группа: %s) ---\n", ref_og))
interpret("Cox PH (время до 1-й повторной операции)", cox_e_og$est, cox_e_og$lo, cox_e_og$hi, cox_e_og$p, ref_og, grp_og)
interpret("Andersen-Gill (все повторные операции)", ag_e_og$est, ag_e_og$lo, ag_e_og$hi, ag_e_og$p, ref_og, grp_og)
interpret("PWP (упорядоченные повторные операции)", pwp_e_og$est, pwp_e_og$lo, pwp_e_og$hi, pwp_e_og$p, ref_og, grp_og)
interpret("Neg. Binomial (общее количество)", nb_e_og$est, nb_e_og$lo, nb_e_og$hi, nb_e_og$p, ref_og, grp_og,
  unit = "повторные операции за единицу времени"
)
print_verdict(list(cox_e_og, ag_e_og, pwp_e_og, nb_e_og))
